# 06 - Forecast-error model, uncertainty reserve and safe envelope (workflow step 7)

**Question:** does the quantile reserve $P_{res,k}$ deliver the stated reliability
$\Pr(\epsilon^{net}_k\le P^{dis}_{res,k})\ge1-\varepsilon$ (and likewise for charging) on data it
was **not** built from, and is the resulting safe set nested,
$\mathcal F^{safe}_k\subseteq\mathcal F^{phys}_k\subseteq\mathcal F^{rated}$?

* Reserve set: 20 000 AR(1) error trajectories (seed 12345) -- used by the EMS.
* Test set: 20 000 independent trajectories (seed 999) -- coverage check only.
* Monte Carlo evaluation set (seed 2026, 100 trials) -- used in notebook 08, never seen by the EMS.

Outputs: `output/results/results_reserve.csv`, `output/results/results_reserve_validation.csv`,
`output/experiment_figures/Fig_uncertainty_reserve.pdf`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import RESULTS_DIR, FIG_DIR
from dfm_acdc.simulate import Study
from dfm_acdc.uncertainty import sample_scenarios, reserve_quantiles, reserve_gaussian_reference
from dfm_acdc.envelope import build_envelope, degradation_allowance
from dfm_acdc import plotstyle

plotstyle.apply_style()
S = Study()
S.load_or_precompute(os.path.join(RESULTS_DIR, "precompute_network_forecast.csv"))
sp, day = S.sp, S.day
EPS = [0.01, 0.05, 0.10, 0.20]
tests = []
def check(name, value, ok, tol=""):
    tests.append({"test": name, "value": value, "tolerance": tol, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {value}")

## Forecast-error distributions

In [ ]:
res_set = sample_scenarios(day, sp.unc, sp.ren.wt_rated_kw, sp.unc.n_samples_reserve, 12345)
test_set = sample_scenarios(day, sp.unc, sp.ren.wt_rated_kw, 20000, 999)
for name, x in (("PV", res_set.eps_pv), ("wind", res_set.eps_wt), ("DC demand", res_set.eps_dc), ("DC net", res_set.eps_net)):
    print(f"{name:10s}: mean {x.mean():6.2f} kW, std (max over day) {x.std(axis=0).max():6.2f} kW, "
          f"P1..P99 at noon [{np.percentile(x[:, 48], 1):.1f}, {np.percentile(x[:, 48], 99):.1f}] kW")
lag1 = np.mean([np.corrcoef(res_set.eps_dc[:, k], res_set.eps_dc[:, k + 1])[0, 1] for k in range(95)])
check("AR(1) lag-1 correlation of the demand error = phi", f"{lag1:.3f} vs {sp.unc.ar1_phi}", abs(lag1 - sp.unc.ar1_phi) < 0.02, "0.02")

## Reserves and out-of-sample coverage

In [ ]:
rows, rv = [], {"t_h": day.t0}
for eps in EPS:
    rd, rc = reserve_quantiles(res_set.eps_net, eps)
    g, s_net = reserve_gaussian_reference(day, sp.unc, sp.ren.wt_rated_kw, eps)
    cov_d = np.mean(test_set.eps_net <= rd[None, :], axis=0)
    cov_c = np.mean(-test_set.eps_net <= rc[None, :], axis=0)
    # sampling s.e. of (estimated quantile vs. independent test set): both sets have 20 000 samples
    se = np.sqrt(2.0 * eps * (1 - eps) / test_set.eps_net.shape[0])
    rv[f"P_res_dis_eps{eps:g}"] = rd; rv[f"P_res_ch_eps{eps:g}"] = rc
    rows.append({"eps": eps, "Pres_dis_mean": rd.mean(), "Pres_dis_max": rd.max(), "Pres_ch_mean": rc.mean(),
                 "Pres_ch_max": rc.max(), "coverage_dis_min": cov_d.min(), "coverage_ch_min": cov_c.min(),
                 "coverage_dis_mean": cov_d.mean(), "coverage_ch_mean": cov_c.mean(), "target": 1 - eps, "max_rel_dev_gauss": np.max(np.abs(rd - g) / np.maximum(g, 1e-9))})
    check(f"eps={eps:g}: mean out-of-sample coverage = 1-eps (+/- 6 s.e./sqrt(96))",
          f"dis {cov_d.mean():.4f}, ch {cov_c.mean():.4f} (target {1-eps:.2f})",
          abs(cov_d.mean() - (1 - eps)) < 2 * se / np.sqrt(96) * 3 and abs(cov_c.mean() - (1 - eps)) < 2 * se / np.sqrt(96) * 3)
    check(f"eps={eps:g}: every interval within 4 combined s.e. of 1-eps",
          f"dis min {cov_d.min():.4f}, ch min {cov_c.min():.4f}",
          cov_d.min() >= 1 - eps - 4 * se and cov_c.min() >= 1 - eps - 4 * se)
rv["sigma_net_kW"] = s_net
pd.DataFrame(rv).to_csv(os.path.join(RESULTS_DIR, "results_reserve.csv"), index=False)
rs = pd.DataFrame(rows)
print(rs.round(4).to_string(index=False))
check("empirical quantile agrees with the Gaussian reference (truncation only)",
      f"max rel. dev. {rs.max_rel_dev_gauss.max():.3f}", rs.max_rel_dev_gauss.max() < 0.08, "8 %")

## Nesting of the safe envelope

In [ ]:
rd, rc = reserve_quantiles(res_set.eps_net, sp.dfm.eps_risk)
b, deg = sp.b, S.deg
F0 = deg.F_from_soh(b.soh_0)
_, D_max, _ = degradation_allowance(deg, F0, 0.0, sp.dfm)
bad = 0
for k in range(96):
    for soc in np.linspace(b.soc_min, b.soc_max, 9):
        for used in (0.0, 0.5, 0.95):
            env = build_envelope(b, deg, sp.dfm, soc, b.soh_0 * b.E_rated_kwh, F0, used * D_max / float(deg.dL(F0)),
                                 S.pre.P_net_dis[k], S.pre.P_net_ch[k], rd[k], rc[k])
            ok = (0 <= env.P_safe_dis <= env.P_flex_dis <= b.P_rated_kw + 1e-9 and
                  0 <= env.P_safe_ch <= env.P_flex_ch <= b.P_rated_kw + 1e-9 and
                  (env.P_safe_dis == 0 or abs(env.P_flex_dis - env.P_safe_dis - rd[k]) < 1e-9))
            bad += not ok
check("F_safe subset F_phys subset F_rated and P_safe = P_flex - P_res when positive (2592 states)", f"{bad} failures", bad == 0)

In [ ]:
val = pd.DataFrame(tests)
val.to_csv(os.path.join(RESULTS_DIR, "results_reserve_validation.csv"), index=False)
print(f"\n{val['pass'].sum()}/{len(val)} tests passed")
assert val["pass"].all()

## Supplementary figure -- DC-subsystem net forecast error and reserves

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_PAGE_WIDTH_IN, 2.5
fig, axs = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))
t = day.t0
ax = axs[0]
q = np.percentile(res_set.eps_net, [1, 5, 25, 75, 95, 99], axis=0)
ax.fill_between(t, q[0], q[5], step="post", color=plotstyle.CAT["blue"], alpha=0.15, lw=0, label="P1-P99")
ax.fill_between(t, q[1], q[4], step="post", color=plotstyle.CAT["blue"], alpha=0.25, lw=0, label="P5-P95")
ax.fill_between(t, q[2], q[3], step="post", color=plotstyle.CAT["blue"], alpha=0.4, lw=0, label="P25-P75")
ax.step(t, rd, where="post", color=plotstyle.CAT["orange"], label=r"$P^{dis}_{res,k}$ ($\varepsilon=0.05$)")
ax.step(t, -rc, where="post", color=plotstyle.CAT["orange"], ls="--", label=r"$-P^{ch}_{res,k}$")
ax.axhline(0, color=plotstyle.INK_MUTED, lw=0.6)
ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 4)); ax.set_xlabel("Time [h]"); ax.set_ylabel(r"$\epsilon^{net}_k$ [kW]")
ax.legend(frameon=False, fontsize=6.0, ncol=2, loc="lower left")
ax = axs[1]
for eps, col in zip(EPS, (plotstyle.CAT["violet"], plotstyle.CAT["orange"], plotstyle.CAT["aqua"], plotstyle.CAT["yellow"])):
    ax.step(t, rv[f"P_res_dis_eps{eps:g}"], where="post", color=col, label=fr"$\varepsilon={eps:g}$")
ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 4)); ax.set_xlabel("Time [h]"); ax.set_ylabel(r"$P^{dis}_{res,k}$ [kW]")
ax.legend(frameon=False, fontsize=6.5)
for a, lab in zip(axs, "ab"):
    a.text(-0.2, 1.04, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_uncertainty_reserve.pdf"))
plt.show()

## Takeaway

The DC-subsystem net forecast error is dominated by PV around midday; the quantile reserve
follows that profile, meets its reliability target out of sample at every interval and for every
risk tolerance, and agrees with the Gaussian reference up to the small effect of truncating PV
at zero. The safe set is nested inside the physical set by construction ($\max\{0,\cdot\}$),
so the uncertainty reserve never changes the physical capability -- it only withholds part of it.